# Task 1 — Wound Region Segmentation: Colab Ablation Run


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the dataset

Unzips the dataset (skipped if already extracted), then finds the
extracted `wound-region-segmentation` folder wherever it landed inside
the zip (its internal layout has drifted before — this searches for it
instead of hardcoding a path) and symlinks it into the repo.

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-region-segmentation.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-region-segmentation"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
print("dataset found at:", source)

os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Ensure the `skin` color mapping matches training code

Forces `skin` to `#0000FF` in the dataset's `color-mappings.json`
(matching `datasets/README.md`) — a no-op if it's already correct.

In [ ]:
import json

color_map_path = f"{DATASET_DIR}/color-mappings.json"
with open(color_map_path) as f:
    mapping = json.load(f)
print("before:", mapping)

if mapping.get("skin") != "#0000FF":
    mapping["skin"] = "#0000FF"
    with open(color_map_path, "w") as f:
        json.dump(mapping, f, indent=4)
    print("fixed ->", mapping)
else:
    print("already correct, no change")

## 5. Install dependencies

In [ ]:
import os
os.chdir(f"{REPO_DIR}/task_1")
!pip install -q -r requirements.txt

## 6. Redirect checkpoints/results to Drive

So a killed/disconnected runtime doesn't lose completed-run history or
checkpoints — everything survives on Drive across sessions.

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/task_1_outputs"

with open("configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/ablation_results.csv"
with open("configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 7. Run the ablation grid

`run_ablation.py` already skips completed `(config, fold)` rows and
resumes a killed fold from its last saved epoch — but a crash (e.g. OOM)
still kills the Python process itself. This loop restarts it
automatically so a crash doesn't need you to re-run the cell by hand.
Output is streamed line-by-line (`Popen` + `-u`) instead of
`subprocess.run()`, which buffers the whole thing and prints nothing
in Colab until the process exits. A full runtime disconnect kills the
loop too (Colab tears down the kernel); at that point just re-run this
cell — the run picks up from the last saved epoch either way.

In [ ]:
!python check_compatibility.py

In [ ]:
import subprocess
import time

cmd = ["python", "-u", "run_ablation.py", "--study", "all", "--batch", "64", "--workers", "8"]
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("grid finished")
        break
    print(f"run_ablation.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 8. Preview sample panels

After every completed `(config, fold)`, `run_ablation.py` saves a
3-panel PNG (image / ground truth / prediction overlay) to
`results/samples/`. This shows the most recent ones.

In [ ]:
import glob
from IPython.display import Image, display

samples_dir = f"{OUTPUT_DIR}/results/samples"
paths = sorted(glob.glob(f"{samples_dir}/*.png"))
print(f"{len(paths)} sample panels")
for path in paths[-10:]:
    print(path.split("/")[-1])
    display(Image(filename=path))